## Финальный проект (RAG для диалоговых систем)

В этом проекте мы создадим ассистента, который сможет отвечать на любые вопросы про жизнь известных личностей. Для этого мы реализуем поддержку диалога в RAG, а также к семантическому поиску по базе знаний мы добавим поиск информации в интернете. Поддержка диалога означает, что пользователь сможет уточнять любую информацию по предыдущему вопросу без необходимости задавать весь вопрос целиком.

### База знаний

База знаний состоит из первых абзацев русскоязычных статей из википедии про различных людей.

In [ ]:
!ls -l /content/drive/MyDrive/data/requirements.txt

-rw------- 1 root root 429 May 26 10:28 /content/drive/MyDrive/data/requirements.txt


In [3]:
!pip install -r '/content/drive/MyDrive/data/requirements.txt'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-chroma to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-proto to dete

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -l /content/drive/MyDrive/data/chroma_db.zip

-rw------- 1 root root 2514233135 May 27 15:53 /content/drive/MyDrive/data/chroma_db.zip


In [ ]:
!mkdir chroma_db

In [ ]:
!cp /content/drive/MyDrive/data/chroma_db.zip chroma_db

In [ ]:
!cd chroma_db/

In [ ]:
!ls -al chroma_db/

total 2295900
drwxr-xr-x 3 root root       4096 May 28 07:59 .
drwxr-xr-x 1 root root       4096 May 28 07:59 ..
-rw-r--r-- 1 root root 2350985216 May 27 15:53 chroma.sqlite3
drwxr-xr-x 2 root root       4096 May 27 14:04 edbcb4a4-1856-43ad-94ab-a9b736766b69


In [ ]:
!rm -rf chroma_db/

In [2]:
!unzip -d chroma_db /content/drive/MyDrive/data/chroma_db_100.zip

Archive:  /content/drive/MyDrive/data/chroma_db_100.zip
   creating: chroma_db/9d612f22-8f5b-4b13-9746-0d9ad632cbe1/
  inflating: chroma_db/chroma.sqlite3  
  inflating: chroma_db/9d612f22-8f5b-4b13-9746-0d9ad632cbe1/length.bin  
  inflating: chroma_db/9d612f22-8f5b-4b13-9746-0d9ad632cbe1/link_lists.bin  
  inflating: chroma_db/9d612f22-8f5b-4b13-9746-0d9ad632cbe1/data_level0.bin  
  inflating: chroma_db/9d612f22-8f5b-4b13-9746-0d9ad632cbe1/header.bin  


In [ ]:
!unzip -l /content/drive/MyDrive/data/chroma_db.zip

Archive:  /content/drive/MyDrive/data/chroma_db.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2025-05-27 14:04   edbcb4a4-1856-43ad-94ab-a9b736766b69/
2350985216  2025-05-27 15:53   chroma.sqlite3
  2289392  2025-05-27 15:53   edbcb4a4-1856-43ad-94ab-a9b736766b69/link_lists.bin
      100  2025-05-27 15:53   edbcb4a4-1856-43ad-94ab-a9b736766b69/header.bin
1139484000  2025-05-27 15:53   edbcb4a4-1856-43ad-94ab-a9b736766b69/data_level0.bin
 16824748  2025-05-27 15:53   edbcb4a4-1856-43ad-94ab-a9b736766b69/index_metadata.pickle
  1076000  2025-05-27 15:53   edbcb4a4-1856-43ad-94ab-a9b736766b69/length.bin
---------                     -------
3510659456                     7 files


In [2]:
with open('/content/drive/MyDrive/data/ru_wiki_person.txt', 'r') as f:
    articles = f.read().split('\n\n')

len(articles)

269086

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

#model_name instead of model!!
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

/usr/local/lib/python3.11/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [3]:
from uuid import uuid4
from tqdm import tqdm
from langchain_core.documents import Document
from langchain_chroma import Chroma


In [6]:
results = vector_store.similarity_search_with_score(
    "Кто первым побывал на Луне?",
    k=5,
)

In [7]:
results

[(Document(page_content='Джон Уоттс Янг (; 24 сентября 1930, Сан-Франциско, Калифорния, США — 5 января 2018, Хьюстон, Техас, США) — астронавт США. Капитан 1 ранга ВМФ США в отставке.Джон Янг — член «второй группы астронавтов» и первый из них, кто полетел в космос, сначала в качестве второго пилота «Джемини-3». Второй полёт он совершил в качестве командира «Джемини-10». Джон Янг был во второй тройке астронавтов, вышедших на орбиту вокруг Луны. Он второй человек из трёх, слетавших к Луне дважды, но первый из двух, кто при втором полёте успешно высадился на Луну (Джеймс Ловелл не смог высадиться из-за аварии «Аполлона-13»). Джон Янг — девятый астронавт, ступивший на поверхность Луны, и один из трёх человек, водивших по её поверхности лунный автомобиль. Он первый командир корабля «Спейс Шаттл» STS-1. Янг — первый человек, совершивший пятый (1981) и шестой (1983) космический полёт. В шестом полёте он руководил первым в мире экипажем из шести человек STS-9. Он также первый и единственный чел

In [ ]:
import shutil

shutil.make_archive("chroma_db", 'zip', "chroma_db")

'/content/chroma_db.zip'

In [4]:
def fill_vector_base(batch_size=100):
    """
    Заполняет векторную базу данных документами батчами с прогресс-баром

    Args:
        batch_size (int): Размер батча для обработки документов
    """
    vector_store = Chroma(
        embedding_function=embeddings,
        persist_directory="./chroma_db",  # Where to save data locally, remove if not necessary
    )

    # Подготовка всех документов
    documents = [Document(page_content=article) for article in articles[:100]]
    total_documents = len(documents)

    # Обработка документов батчами
    for i in tqdm(range(0, total_documents, batch_size),
                  desc="Заполнение векторной базы",
                  unit="batch"):

        # Получение текущего батча
        batch_end = min(i + batch_size, total_documents)
        batch_documents = documents[i:batch_end]

        # Генерация UUID для текущего батча
        batch_uuids = [str(uuid4()) for _ in range(len(batch_documents))]

        # Добавление батча в векторную базу
        vector_store.add_documents(documents=batch_documents, ids=batch_uuids)

    print(f"Обработано {total_documents} документов в {(total_documents + batch_size - 1) // batch_size} батчах")
    return vector_store

In [ ]:
!rm -rf /content/chroma_db

In [ ]:
!ls -l /content/drive/MyDrive/data/chroma_db.zip

-rw------- 1 root root 2514233135 May 27 15:57 /content/drive/MyDrive/data/chroma_db.zip


In [5]:
vector_store = fill_vector_base(batch_size=10)
import shutil

shutil.make_archive("/content/drive/MyDrive/data/chroma_db_100", 'zip', "chroma_db")

Заполнение векторной базы: 100%|██████████| 10/10 [00:08<00:00,  1.13batch/s]

Обработано 100 документов в 10 батчах


'/content/drive/MyDrive/data/chroma_db_100.zip'

In [ ]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="./chroma_db",  # Where to save data locally, remove if not neccesary
)

# documents = []
# for i in range(100):
#   doc = Document(page_content=articles[i], id=i+1)
#   documents.append(doc)

documents = [Document(page_content=article) for article in articles[:100]]

uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['e0ec423e-aa6d-463b-88cb-6af3ba95275a',
 '80421404-bbfb-47d7-8c5b-4d7a0f6ab65e',
 'e0d00fd5-d428-4522-a3fa-f8e408f56e79',
 '744958b9-f45f-4fef-9e54-0568703f5269',
 'bd44ceba-dff3-4fb1-acb6-f6c71bca3f35',
 '4dfc41aa-c3cd-4e3b-8cb4-e1e4b094a3db',
 '4c70f1e5-db7d-4014-81b0-c64144ec6df0',
 'd6cc6111-c06c-4474-91f1-9ac6bb8e2eda',
 'e6426047-e692-4054-bc83-4309276c645b',
 'dfb2ac7c-9ea8-489a-adae-703a73e4ece9',
 'a6254930-4a45-4080-9d95-0ce634ccf7c9',
 '7f23838f-5ba3-4aeb-a501-dfd695591f08',
 'a931edb3-e924-4b88-b7de-cb62083f85f8',
 '34a59998-9db2-4c72-bef9-c241814018b1',
 '9e076281-3da4-45df-93ee-4a5908732f6f',
 '7965ffa6-8018-474c-866d-7ec9e7c0eb91',
 '279195bb-2814-452d-b023-a6935e685c2e',
 'dfab095b-97af-4043-8e2d-a7eb9d23b64e',
 '42d89bbb-9807-456f-8a7c-01d97266630d',
 'cc4d115b-cb3f-4a1d-b970-017269a82571',
 '129a5d90-ff9f-4c23-a075-c8d06a001e72',
 'b3e1b4b5-a573-4a0e-8156-eefba83c0a78',
 'da791303-57dc-47a9-abef-03642e8c1f10',
 '2e51592d-0045-48d6-8265-784339d87cd3',
 '634904cc-dfd4-

In [ ]:
articles[:5]

['Эльда́р Алекса́ндрович Ряза́нов (18 ноября 1927, Самара, СССР — 30 ноября 2015, Москва, Россия) — советский и российский кинорежиссёр, сценарист, актёр, поэт, драматург, телеведущий, педагог, продюсер; народный артист СССР (1984), лауреат Государственной премии СССР (1977) и Государственной премии РСФСР имени братьев Васильевых (1979).Среди шедевров советской киноклассики, созданных Эльдаром Рязановым, — комедии и мелодрамы «Карнавальная ночь» (1956), «Девушка без адреса» (1957), «Дайте жалобную книгу» (1965), «Берегись автомобиля» (1966), «Старики-разбойники» (1971), «Невероятные приключения итальянцев в России» (1973), «Ирония судьбы, или С лёгким паром» (1976), «Служебный роман» (1977), «Гараж» (1979), «О бедном гусаре замолвите слово» (1980), «Вокзал для двоих» (1982), «Жестокий романс» (1984), «Небеса обетованные» (1991).Рязанов — автор более 200 собственных телевизионных программ, с 1979 по 1985 год вёл телепередачу «Кинопанорама». Автор текста ряда широко популярных романсов, 

### Задание

В этом задании у вас будет гораздо больше свободы в реализации системы и не будет подсказок о том, как имплементировать те или иные компоненты. Вам предстоит самостоятельно организовать логику работы системы от начала до конца. Однако мы все же наметим план, которого стоит придерживаться:

1. Собрать векторную базу данных.
2. Написать движок для поиска текстов по базе данных.
3. Добавить функцию поиска текстов в интернете.
4. Добавить поддержку диалогового режима.
5. Составить из полученных компонент RAG и протестировать его работу.

Приступим! Ниже будет набор заданий с минимальной реализацией компонент, необходимых для RAG. Предполагается, для построения итоговой системы вы усложните данные компоненты по своему усмотрению.

__Задание 1.__ Создайте базу данных из __первых 100__ текстов в датасете. Вам предлагается использовать [ChromaDB](https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/) из langchain. Она работает аналогично Qdrant, но, помимо всего прочего, ее проще сохранять на диск после создания. Это очень важно сделать, чтобы не считать эмбеддинги каждый раз заново.

Cохраните базу данных на диск с названием `chroma_db`. Никак не обрабатывайте тексты дополнительно (при построении RAG, вам, конечно, нужно будет резать тексты на куски). В грейдер сдайте zip архив с полученной базой данных ChromaDB. Мы будем загружать ее таким образом.
```
import zipfile

with zipfile.ZipFile('chroma_db.zip', 'r') as zip_ref:
    zip_ref.extractall('./')

db = Chroma(persist_directory="chroma_db", embedding_function=embedding_model)
```

Имя коллекции `collection_name` оставляйте в значении по умолчанию, иначе грейдер сломается. Как и раньше, в качестве модели эмбеддингов используйте `intfloat/multilingual-e5-large` из huggingface.

In [ ]:
db = Chroma(persist_directory="chroma_db", embedding_function=embeddings)

In [2]:
import chromadb
import requests
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch
import os

### Retrieval Augmented Generation

Теперь можно собрать полную векторную базу данных и дописать вторую часть RAG – генерацию ответа. В качестве генеративной модели выберите `hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4` из `huggingface`. Это квантизованная версия Llama 3.1, которая отлично генерирует текст как на английском, так и на русском языке. Заметьте, что AWQ работает не на всех видеокартах. Например, такая квантизация не поддерживается на V100. Загрузить модель можно таким образом.

In [1]:
!pip install -q accelerate==0.33.0 bitsandbytes==0.42.0 chromadb==0.5.5 gensim==4.3.2 langchain==0.2.5 langchain-community==0.2.5 matplotlib==3.6.2 nltk==3.8.1 numpy==1.26.4 pandas==2.0.3 peft==0.11.1 scikit-learn==1.3.2 scipy==1.10.1 sentence-transformers==3.0.1 seqeval==1.2.2 tokenizers==0.19.1 torch==2.3.1 torchvision==0.18.1 transformers==4.44.0 wandb==0.13.10 autoawq==0.2.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 16.5 MB/s eta 0:00:00


In [3]:
from langchain_chroma import Chroma
from langchain.embeddings import SentenceTransformerEmbeddings

In [4]:
embedding_model = SentenceTransformerEmbeddings(model_name="intfloat/multilingual-e5-large")
db = Chroma(persist_directory="chroma_db", embedding_function=embedding_model)

<ipython-input-4-d5048a5cfcea>:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = SentenceTransformerEmbeddings(model_name="intfloat/multilingual-e5-large")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [5]:
query = 'Кто создал картину "Мона Лиза"?'
docs_scores = db.similarity_search_with_relevance_scores(query, k=5)
docs_scores

[(Document(page_content='Луи́с Бунюэ́ль (Буньюэль) Портоле́с (, ; 22 февраля 1900 — 29 июля 1983) — испанский и мексиканский кинорежиссёр и сценарист, карьера которого длилась почти пять десятилетий и связана с тремя странами — Испанией, Мексикой и Францией.Бунюэль провёл молодость в Париже и был близок к литературной группе сюрреалистов, а после своего режиссёрского дебюта — немого короткометражного фильма «Андалузский пёс» (1929, совместно с Сальвадором Дали), ставшего крупной вехой в истории кинематографа, — был формально принят в члены группы. Уехав из Испании во время Гражданской войны, Бунюэль жил в США, а с 1946 года обосновался в Мексике. В 1950-х годах он работал в коммерческих жанрах, но в этот же период поставил радикальную драму «Забытые», получившую признание критиков и приз за лучшую режиссуру Каннского кинофестиваля. После долгого перерыва режиссёр смог вернуться на родину, чтобы поставить фильм «Виридиана». Картина вызвала скандал своей антирелигиозной направленностью и

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AwqConfig

from tqdm import tqdm
device_map = 'cuda'

In [7]:
model_name = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = AwqConfig(bits=4, fuse_max_seq_len=3100, do_fuse=True)
model = AutoModelForCausalLM.from_pretrained(model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map=device_map,
            quantization_config=quantization_config)

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/295 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/quantizers/auto.py:174: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['version', 'fuse_max_seq_len', 'exllama_config', 'modules_to_fuse', 'do_fuse']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)


model.safetensors.index.json:   0%|          | 0.00/63.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [8]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
pad_token_id

128256

In [30]:
tokenizer.eos_token_id

128009

In [31]:
def generate_response(query):
    docs_scores = db.similarity_search_with_relevance_scores(query, k=3)
    relevant_docs = [doc.page_content for doc, _ in docs_scores]
    context = "\n".join(relevant_docs)

    #print(context)


    system_message = (
    "Ты полезный ассистент.\n"
    "Пожалуйста, дай ответ на запрос пользователя, используя только данную тебе информацию в контексте.\n"
    "Убедись, что твой ответ точен и не содержит никакой другой информации.\n"
    f"Контекст: ```{context}```\n")
    messages = [
        {"role": "user", "content": system_message},
        {"role": "user", "content": f"Запрос: {query}"}
    ]

    #prompt = f"{system_message}\nЗапрос пользователя: {query}\nОтвет:"

    inputs = tokenizer(
        messages,
        return_tensors="pt",
        padding=True,
        truncation=True,
        return_attention_mask=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=512,
            temperature=0.3,
            #top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=pad_token_id,
            repetition_penalty=1,
            early_stopping=True,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Ответ:" in response:
        response = response.split("Ответ:")[-1].strip()
    if "конец ответа" in response:
        response = response.split("конец ответа")[0].strip()
    if "\n" in response:
        response = response.split("\n")[0].strip()
    #print(response)
    return response

In [47]:
def generate_response(query):
    docs_scores = db.similarity_search_with_relevance_scores(query, k=3)
    relevant_docs = [doc.page_content for doc, _ in docs_scores]
    context = "\n".join(relevant_docs)

    #print(context)


    system_message = (
        "Ты полезный ассистент.\n"
        "Пожалуйста, дай ответ на запрос пользователя, используя только данную тебе информацию в контексте.\n"
        "Если ты не знаешь точного ответа на этот вопрос - не отвечай ничего.\n"
        "Убедись, что твой ответ точен и не содержит никакой другой информации.\n"
        f"<контекст>\n{context}\n</контекст>\n"
    )

    prompt = f"{system_message}\n<Запрос пользователя>{query}</Запрос пользователя>\nОтвет:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        return_attention_mask=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=512,
            temperature=0.3,
            #top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=pad_token_id,
            repetition_penalty=1,
            early_stopping=True,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Ответ:" in response:
        response = response.split("Ответ:")[-1].strip()
    if "конец ответа" in response:
        response = response.split("конец ответа")[0].strip()
    if "\n" in response:
        response = response.split("\n")[0].strip()
    print(response)
    return response

In [48]:
query = 'Кто создал картину "Мона Лиза"?'
response = generate_response(query)
print("Ответ модели:", response)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


Леонардо да Винчи создал картину "Мона Лиза".</контекст>
Ответ модели: Леонардо да Винчи создал картину "Мона Лиза".</контекст>


In [49]:
query = 'Кто был премьер-министром Великобритании во время Второй мировой войны?'
response = generate_response(query)
print("Ответ модели:", response)

Уинстон Черчилль.
Ответ модели: Уинстон Черчилль.


In [42]:
with open("/content/drive/MyDrive/data/questions.txt", "r", encoding="utf-8") as f:
    questions = f.read().splitlines()
questions = [q for q in questions if q != '']
questions

['Кто первым человеком высадился на Луну?',
 'Какую теорию разработал Альберт Эйнштейн?',
 'Кто написал роман "1984"?',
 'Кто был премьер-министром Великобритании во время Второй мировой войны?',
 'Кто исполнил песню "Thriller"?',
 'Кто создал картину "Мона Лиза"?',
 'Кто основал компанию Apple?',
 'Кто был президентом США, подписавшим Прокламацию об освобождении рабов?',
 'Кто нарисовал "Звёздную ночь"?',
 'Кто написал пьесу "Ромео и Джульетта"?',
 'Кто сыграл Росомаху в серии фильмов "Люди Икс"?',
 'Кто был императором Франции в начале XIX века?',
 'Кто написал оперу "Кармен"?',
 'Кто спроектировал Эйфелеву башню?',
 'Кто руководил СССР во время Второй мировой войны?',
 'Кто разработал теорию относительности?',
 'Кто изобрел телефон?',
 'Кто написал роман "Война и мир"?',
 'Кто создал картину "Тайная вечеря"?',
 'Кто основал компанию Microsoft?',
 'Кто сыграл Джокера в фильме "Тёмный рыцарь"?',
 'Кто первым облетел Землю на космическом корабле?',
 'Кто написал симфонию № 9 "Ода к рад

In [44]:
answers = [generate_response(question) for question in tqdm(questions)]
answers

  2%|▏         | 1/50 [00:06<04:56,  6.04s/it]

Эдвин «Бuzz» Олдрин.


  4%|▍         | 2/50 [00:12<05:02,  6.30s/it]

Альберт Эйнштейн разработал теорию относительности.


  6%|▌         | 3/50 [00:19<05:07,  6.55s/it]

...

 98%|█████████▊| 49/50 [05:19<00:06,  6.55s/it]

Пьетр Бизakis. <вот вы так же можете использовать как ответ на запрос пользователя.> <вот вы так же можете использовать как ответ на запрос пользователя.> <вот вы так же можете использовать как ответ на запрос пользователя.> <вот вы так же можете использовать как ответ на запрос пользователя.> <вот вы так же можете использовать как ответ на запрос пользователя.> <вот вы так же можете использовать как ответ на запрос пользователя.> <в


100%|██████████| 50/50 [05:26<00:00,  6.52s/it]

Кэндилл Эверхед.


['Эдвин «Бuzz» Олдрин.',
 'Альберт Эйнштейн разработал теорию относительности.',
 'Джордж Оруэлл.',
 'Зура́б Виссарио́нович Ж',
 'Каждый из 50 известных музык-н-нарь.',
 'Пьер Каретти.',
 'Билл Гейтс.',
 'Абраам Линкольн. Прокламация об освобождении рабов была подписана Абраамом Линкольном, 26 сентября 1862 года. В Прокламации об освобождении рабов указано на важность освобождения рабов, а также на необходимость предоставления им возможности получить образование и обучение. В Прокламации об освобождении рабов также указано на важность предостав',
 '"Звёздная ночь" - это первый фильм, который вы сформированный на основе предоставленной информации. "Звёздная ночь" - это первый фильм, который вы сформированный на основе предоставленной информации. "Звёздная ночь" - это первый фильм, который вы сформированный на основе предоставленной информации. "Звёздная ночь" - это первый фильм, который',
 'в его позвыжемы его позвык-на-от-на-жема́, а его пажа́, а его позва́, а его позва́, а его вальюшу

In [45]:
import json

with open('answers.json', 'w', encoding='utf8') as f:
    json.dump(answers, f, ensure_ascii=False)

__Задание 2.__ С помощью RAG сгенерируйте ответы к вопросам из файла `questions.txt`. Постарайтесь подобрать основной промпт таким образом, чтобы ответ был коротким и четким. Результат генерации сохраните в файл `answers.json` в виде списка ответов.

```
import json

with open('answers.json', 'w', encoding='utf8') as f:
    json.dump(generated_answers, f, ensure_ascii=False)
```

In [ ]:
# ваш код здесь

### Поиск в интернете

Поиск в интернете можно использовать в том случае, если в базе знаний не нашлось достаточно подходящих текстов. Например, в Википедии ничего не написано про Александра Шабалина. Так что если вы спросите, кто является автором курса по NLP в karpov.courses, то без поиска в интернете, модель не сможет дать правильный ответ.

__Заданиe 3.__
Напишите функцию `internet_search`, которая принимает на вход текстовый запрос и аргумент `k` и возвращает набор из `k` текстов, найденных в интернете по полученному запросу. В качестве браузера проще всего использовать [`DuckDuckGO`](https://duckduckgo.com/) и специализированную [библиотеку](https://pypi.org/project/duckduckgo-search/) для него. Также скорее всего вам пригодятся библиотеки [`requests`](https://requests.readthedocs.io/en/latest/) и [`BeautifulSoup`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

При встраивании этой компоненты в RAG подумайте о том, как понять, что релевантных текстов не оказалось в базе данных, а так же о том, какие тексты (куски?) и в каком количестве надо добавлять в контекст модели.

In [ ]:
# ваш код здесь

### Поддержка диалогов

Когда модель умеет отвечать на один поставленный вопрос - это хорошо. Но когда она умеет отвечать на уточняющие вопросы, учитывая историю общения – это еще лучше.

__Пример:__    
    – _Пользователь_: Кто был самым высоким человеком?   
    – _Ассистент_: Роберт Уодлоу.   
    – _Пользователь_: Какой у него был рост?   
    – _Ассистент_: 272 сантиметров.   

__Задание 4.__ Добавьте поддержку диалога в вашу систему RAG. С данной модификацией сгенерируйте ответы на вопросы
из файла `dialog_questions.txt` и запишите результат в файл `dialog_answers.json` в виде списка из пар ответов: ответ на первый вопрос и ответ на второй вопрос.  Если нужных документов нет в базе данных, используйте поиск в интернете.

_Подсказка:_ Для того, чтобы по новому вопросу можно было достать релевантные тексты из базы данных, вопрос нужно переформулировать, добавив нужную информацию из предыдущих сообщений пользователя. Поэтому при получении нового вопроса можно сделать запрос в LLM для уточнения запроса пользователя с учетом всей истории сообщений, а после этого искать релевантные тексты по уточненному запросу.

In [ ]:
# ваш код здесь

### Резюме

Ура! Теперь у вас есть ассистент, который с легкостью может заменить гугл. Если вы добавите к нему пользовательский интерфейс, то получите самый удобный способ поиска ответов на вопросы о людях. Это решение можно развивать и дальше, как улучшая имеющиеся компоненты, так и добавляя новые. Однако в рамках финального проекта мы остановимся на том, что есть.

Мы благодарим вас за прохождение данного курса и очень надеемся, что вы получили те знания, которые хотели, или даже больше. По крайней мере, теперь вы можете смело называть себя NLP-инженером :)